# DeepDoc full RAG trên Colab
**PDF → native parser/OCR → evidence → Canonical v2 → PublishPolicy → chunking → SQLite/BM25 → citation.**

Chọn GPU runtime, điền hai đường dẫn, rồi Run all. Notebook dùng code repo đã đồng bộ Drive.
Native text tốt bỏ qua OCR. Trang/vùng cần OCR dùng Tesseract Việt/Anh; PP-Structure xử lý cấu trúc và FormulaNet-S xử lý công thức. **Không nạp OCR-VL.**
Đây là full pipeline tài liệu của repo với local RAG harness hiện có, không phải toàn bộ web/Core/database và chưa có bước LLM sinh câu trả lời.


In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,subprocess,shutil,time,signal,datetime,hashlib
RAG_DRIVE = '/content/drive/MyDrive/rag'
INPUT_DRIVE = '/content/drive/MyDrive/CanhDieu/SBT Toan 10 - Tap 1.pdf' # FILE PDF thật trên Drive
QUERY = 'Nội dung chính của bài học là gì?'
DPI = 144
MAX_PIXELS = 1_000_000
PAGE_TIMEOUT = 600
DOCUMENT_TIMEOUT = 14400 # 4 giờ; vẫn chịu giới hạn tài nguyên admission của repo
GPU_INDEX = 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'


Mounted at /content/drive


## Môi trường
Hai venv Linux riêng: core giữ requirements local; GPU dùng Paddle. Tách riêng để tránh xung đột numpy/protobuf.
Venv ở `/content`; pip cache và model cache trên Drive. Phiên Colab mới vẫn cần tạo venv, không dùng venv Windows.


In [2]:
rag=Path(RAG_DRIVE).resolve()
if not INPUT_DRIVE: raise ValueError('Điền INPUT_DRIVE tới FILE PDF ở ô đầu.')
source=Path(INPUT_DRIVE).resolve()
assert source.is_file() and source.suffix.lower()=='.pdf',source
for file in ('idp/pdf/service.py','colab/full_rag.py','colab/structure_worker.py','requirements-phase0.lock.txt'):
    assert (rag/file).is_file(),f'Chờ Drive đồng bộ file {file}'
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],capture_output=True,text=True)
if gpu.returncode: raise RuntimeError('Chọn GPU runtime. '+gpu.stderr)
print(gpu.stdout)
driver=gpu.stdout.splitlines()[0].split(',')[1].strip()
if 'cu126' in GPU_INDEX and tuple(map(int,driver.split('.')))<(550,54,14): raise RuntimeError('Driver chưa đủ cho cu126.')
assert 72<=DPI<=300 and 1<=MAX_PIXELS<=4_000_000 and PAGE_TIMEOUT>0 and DOCUMENT_TIMEOUT>0
work=Path('/content/deepdoc-full');work.mkdir(exist_ok=True)
cache=Path('/content/drive/MyDrive/deepdoc-colab-cache');cache.mkdir(exist_ok=True)
os.environ['PIP_CACHE_DIR']=str(cache/'pip')
subprocess.run(['apt-get','update','-qq'],check=True)
subprocess.run(['apt-get','install','-y','-qq','poppler-utils','tesseract-ocr','tesseract-ocr-eng','tesseract-ocr-vie','fonts-dejavu-core','libglib2.0-0','libgomp1','libgl1'],check=True)
def make_venv(name):
    path=work/name;py=path/'bin/python'
    if not py.is_file():
        r=subprocess.run([sys.executable,'-m','venv',str(path)],capture_output=True,text=True)
        if r.returncode:
            bootstrap=work/'bootstrap'
            subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'virtualenv'],check=True)
            subprocess.run([sys.executable,'-m','virtualenv',str(path)],env={**os.environ,'PYTHONPATH':str(bootstrap)},check=True)
    return py
core_py=make_venv('core-venv');model_py=make_venv('model-venv')
lock=rag/'requirements-phase0.lock.txt'
core_stamp=hashlib.sha256(lock.read_bytes()+sys.version.encode()).hexdigest()
core_mark=work/'core-installed.txt'
if not core_mark.is_file() or core_mark.read_text()!=core_stamp:
    subprocess.run([str(core_py),'-m','pip','install','-r',str(lock)],check=True)
    subprocess.run([str(core_py),'-m','pip','check'],check=True)
    core_mark.write_text(core_stamp)
model_req=['paddleocr[doc-parser]==3.7.0','paddlex==3.7.2','sentencepiece==0.2.1','protobuf==6.33.4']
model_stamp=hashlib.sha256(json.dumps([model_req,sys.version,GPU_INDEX,'paddlepaddle-gpu==3.3.1']).encode()).hexdigest()
model_mark=work/'models-installed.txt'
if not model_mark.is_file() or model_mark.read_text()!=model_stamp:
    subprocess.run([str(model_py),'-m','pip','install','paddlepaddle-gpu==3.3.1','--index-url',GPU_INDEX],check=True)
    subprocess.run([str(model_py),'-m','pip','install',*model_req],check=True)
    subprocess.run([str(model_py),'-m','pip','check'],check=True)
    model_mark.write_text(model_stamp)
subprocess.run([str(model_py),'-c','from sentencepiece import sentencepiece_model_pb2; from paddleocr import PPStructureV3; import paddle; assert paddle.is_compiled_with_cuda(); assert paddle.device.cuda.device_count()>0; print("Paddle/Tokenizer/CUDA PASS")'],check=True)
print('Hai môi trường đã sẵn sàng')


NVIDIA L4, 580.82.07, 23034 MiB

Hai môi trường đã sẵn sàng


In [3]:
# Snapshot code cần thiết, không copy .venv/.git/dữ liệu riêng tư.
snapshot=work/'repo';snapshot.mkdir(exist_ok=True)
for folder in ('idp','parser','chunking','search','retrieval','colab'):
    shutil.copytree(rag/folder,snapshot/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__','.venv','*.ipynb'))
def sync(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists():return
    for p in src.rglob('*'):
        if 'PaddleOCR-VL' in str(p):continue
        if not p.is_file() or p.name.endswith(('.lock','.incomplete','.tmp')):continue
        target=dst/p.relative_to(src)
        if target.is_file() and target.stat().st_size==p.stat().st_size and target.stat().st_mtime>=p.stat().st_mtime:continue
        target.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,target)
home_local=work/'model-home';home_local.mkdir(exist_ok=True)
print('Staging model cache...')
sync(cache/'model-home/.paddlex',home_local/'.paddlex')
sync(cache/'model-home/.cache',home_local/'.cache')
local_pdf=work/'input.pdf';shutil.copy2(source,local_pdf)
attempt=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S')+'-'+os.urandom(3).hex()
out=Path('/content/drive/MyDrive/deepdoc-full-rag-results')/attempt;out.mkdir(parents=True)
model_env={'HOME':str(home_local),'HF_HUB_DISABLE_XET':'1','PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK':'True','OMP_NUM_THREADS':'1','TOKENIZERS_PARALLELISM':'false','PYTHONUNBUFFERED':'1'}
config={'input':str(local_pdf),'output':str(out),'work':str(work),'model_python':str(model_py),
        'gpu_worker':str(snapshot/'colab/structure_worker.py'),'source_worker':str(snapshot/'idp/ocr/backends/model_worker.py'),
        'model_env':model_env,'dpi':DPI,'max_pixels':MAX_PIXELS,'page_timeout':PAGE_TIMEOUT,
        'document_timeout':DOCUMENT_TIMEOUT,'query':QUERY}
config_path=work/'job.json';config_path.write_text(json.dumps(config),encoding='utf-8')
shutil.copy2(config_path,out/'config.json')
for name,py in [('core',core_py),('model',model_py)]:
    (out/(name+'-environment.txt')).write_text(subprocess.check_output([str(py),'-m','pip','freeze'],text=True),encoding='utf-8')
env={**os.environ,'PYTHONPATH':str(snapshot),'PYTHONUNBUFFERED':'1'}
subprocess.run([str(core_py),'-c','from idp.pdf.service import DeepDocService; from retrieval.canonical_harness import LocalRagIndex; print("Core/Chunker/Retrieval imports PASS")'],env=env,check=True)
print('Full pipeline output:',out)


Staging model cache...
Full pipeline output: /content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a


## Chạy full pipeline
GPU PP-Structure chỉ nạp khi router gặp trang/vùng cần OCR; model giữ trong worker cho cả tài liệu.
Trang native tốt có thể không dùng GPU. Bảng/công thức model chưa xác minh được giữ trong L1/Canonical và bị PublishPolicy chặn khi cần; không nới policy để tăng số chunk.
Notebook chạy RAG harness hiện có: SQLite + BM25 + citation, chưa phải production permission backend hoặc vector/LLM QA.


In [4]:
log_path=out/'pipeline.log';proc=None;error=None
try:
    with log_path.open('w',encoding='utf-8') as log:
        proc=subprocess.Popen([str(core_py),str(snapshot/'colab/full_rag.py'),'--config',str(config_path)],env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        started=time.monotonic()
        while proc.poll() is None:
            if time.monotonic()-started>DOCUMENT_TIMEOUT+60:raise TimeoutError('Full document timeout')
            print('Đang chạy',round(time.monotonic()-started),'s;',subprocess.check_output(['nvidia-smi','--query-gpu=memory.used,memory.total,utilization.gpu','--format=csv,noheader,nounits'],text=True).strip(),flush=True)
            time.sleep(10)
        if proc.wait()!=0:raise RuntimeError('Pipeline exit '+str(proc.returncode))
except BaseException as exc:error=f'{type(exc).__name__}: {exc}'
finally:
    if proc is not None and proc.poll() is None:
        try:os.killpg(proc.pid,signal.SIGKILL)
        except ProcessLookupError:pass
        proc.wait()
    print(log_path.read_text(encoding='utf-8',errors='replace')[-10000:])
    if (out/'structure-worker.log').exists():print((out/'structure-worker.log').read_text(encoding='utf-8',errors='replace')[-5000:])
if error:raise RuntimeError(error)
print('Saving model cache...')
sync(home_local/'.paddlex',cache/'model-home/.paddlex');sync(home_local/'.cache',cache/'model-home/.cache')
summary=json.loads((out/'summary.json').read_text(encoding='utf-8'));print(json.dumps(summary,ensure_ascii=False,indent=2))


Đang chạy 0 s; 0, 23034, 0
Đang chạy 10 s; 372, 23034, 0
Đang chạy 20 s; 618, 23034, 0
Đang chạy 30 s; 712, 23034, 0
Đang chạy 40 s; 1296, 23034, 0
Đang chạy 50 s; 1448, 23034, 0
Đang chạy 60 s; 1712, 23034, 0
Đang chạy 70 s; 2894, 23034, 0
Đang chạy 80 s; 3140, 23034, 0
Đang chạy 90 s; 3140, 23034, 0
Đang chạy 100 s; 3140, 23034, 72
Đang chạy 110 s; 3140, 23034, 8
Đang chạy 120 s; 3140, 23034, 12
Đang chạy 130 s; 3140, 23034, 21
Đang chạy 140 s; 3140, 23034, 0
Đang chạy 150 s; 3140, 23034, 0
Đang chạy 160 s; 3140, 23034, 0
Đang chạy 170 s; 3140, 23034, 0
Đang chạy 180 s; 3140, 23034, 16
Đang chạy 190 s; 3140, 23034, 5
Đang chạy 201 s; 3140, 23034, 0
Đang chạy 211 s; 3140, 23034, 20
Đang chạy 221 s; 3140, 23034, 62
Đang chạy 231 s; 3140, 23034, 0
Đang chạy 241 s; 3140, 23034, 0
Đang chạy 251 s; 3140, 23034, 0
Đang chạy 261 s; 3140, 23034, 0
Đang chạy 271 s; 3386, 23034, 0
Đang chạy 281 s; 3386, 23034, 0
Đang chạy 291 s; 3386, 23034, 14
Đang chạy 301 s; 3388, 23034, 32
Đang chạy 311 s; 

In [5]:
print('Canonical:',out/'deepdoc/canonical.json')
print('L1 evidence:',out/'deepdoc/extraction.json')
print('Chunks:',out/'chunks.json')
print('Index:',out/'rag.sqlite3')
results=json.loads((out/'retrieval.json').read_text(encoding='utf-8'))
for i,result in enumerate(results,1):
    print('HIT',i,result['hit']['text'][:1500])
    print('Citations:',json.dumps(result['citations'],ensure_ascii=False)[:1500])
if not results:print('Không có hit: kiểm tra query, publication.json và eligible/indexed counts; không tự publish nội dung đang review.')


Canonical: /content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a/deepdoc/canonical.json
L1 evidence: /content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a/deepdoc/extraction.json
Chunks: /content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a/chunks.json
Index: /content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a/rag.sqlite3
HIT 1 Các bài tập cơ bản gồm những bài tập giúp học sinh củng cố, kết nỗi các
Citations: [{"block_id": "376a70c666a0fe41e6ef77db4e4598d0981362f87c12d6af03767eb67bf56cb2", "source_id": "ocr:3043100ba8bd6bf14ce838aed37c7e9b5e4d8f30ff83992c8e0b29890fb42ef6", "region_id": "region:e959e41baa0702352e9186a7ff52c3b73d9238bb015be944ec4dd1328c2e2196", "candidate_id": "candidate:a8129e3e6a50590321da628cee845895b03419f97e9a83b6f0ccce6c531ca55f", "evidence_refs": ["evidence:bde4dfcd326246008fee8cd5a7ad76e4710856b3383f8815ae2983b1db4b7a5d"], "page": 4, "bbox": [113.95966706302022, 343.12323529411765, 485.56727705

In [6]:
# Sửa câu hỏi rồi chạy lại cell này; không cần OCR lại tài liệu.
QUERY_AGAIN = QUERY
query_code = r"""
import json,sys
from pathlib import Path
from retrieval.canonical_harness import LocalRagIndex,resolve_citation
root=Path(sys.argv[1]);doc=json.loads((root/'deepdoc/canonical.json').read_text(encoding='utf-8'))
allow=lambda actor,sha,ver: actor=='colab-local-owner' and sha==doc['source_sha256'] and ver==doc['document_version']
hits=LocalRagIndex(root/'rag.sqlite3').retrieve(sys.argv[2],actor='colab-local-owner',access_check=allow,top_k=5)
for hit in hits:
    print(hit['text']);print(json.dumps(resolve_citation(hit,doc),ensure_ascii=False))
print('Hits:',len(hits))
"""
subprocess.run([str(core_py),'-c',query_code,str(out),QUERY_AGAIN],env=env,check=True)


CompletedProcess(args=['/content/deepdoc-full/core-venv/bin/python', '-c', "\nimport json,sys\nfrom pathlib import Path\nfrom retrieval.canonical_harness import LocalRagIndex,resolve_citation\nroot=Path(sys.argv[1]);doc=json.loads((root/'deepdoc/canonical.json').read_text(encoding='utf-8'))\nallow=lambda actor,sha,ver: actor=='colab-local-owner' and sha==doc['source_sha256'] and ver==doc['document_version']\nhits=LocalRagIndex(root/'rag.sqlite3').retrieve(sys.argv[2],actor='colab-local-owner',access_check=allow,top_k=5)\nfor hit in hits:\n    print(hit['text']);print(json.dumps(resolve_citation(hit,doc),ensure_ascii=False))\nprint('Hits:',len(hits))\n", '/content/drive/MyDrive/deepdoc-full-rag-results/20261117T152052-51036a', 'Nội dung chính của bài học là gì?'], returncode=0)

## Điều cần biết
- Notebook mới này gọi pipeline local thật; notebook OCR-VL cũ chỉ dành cho chạy riêng backend đó.
- Full service hiện finalize ở mức document; không có resume từng trang như notebook raw OCR-VL. Kết quả run được tách theo attempt, index/query có thể dùng lại sau khi pipeline thành công.
- Theo admission hiện có, PDF tối đa 256 MiB/2000 trang và tổng render budget 256 triệu pixel. Vượt giới hạn sẽ dừng rõ ràng, không bỏ trang. Có thể hạ độ phân giải bằng cấu hình nhưng cần đánh giá chất lượng.
- Công thức và bảng ở trạng thái unverified không tự thành source proof. Text native hoặc OCR đủ điều kiện mới vào index.
- Phiên bản notebook này chưa chạy GPU Colab khi bàn giao. Không cam kết tốc độ/chất lượng trước khi đo trên tài liệu thật.
- Bản full pipeline dùng CPU/GPU venv riêng; không sửa production default hoặc database.

[PP-StructureV3](https://www.paddleocr.ai/v3.3.1/en/version3.x/pipeline_usage/PP-StructureV3.html) · [FormulaNet nhỏ](https://www.paddleocr.ai/v3.5.0/en/version3.x/pipeline_usage/formula_recognition.html)
